In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from openai import OpenAI
from sklearn.decomposition import PCA
import tiktoken
import getpass
import os

# API Key Setup
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

client = OpenAI()
def get_embedding(text, model='text-embedding-3-small'):              #<<----------this is embedding model 
    text_clean = text.replace('\n', ' ')

    resp = client.embeddings.create(input=[text_clean], model=model)
    return np.array(resp.data[0].embedding)

sample = 'The quick brown fox jumps over the lazy dog.'
emb = get_embedding(sample)
print(f'Vector length: {len(emb)}')
print('First 5 dims:', emb[:5])

In [ ]:
# Vector length: 1536
# First 5 dims: [-0.01842354 -0.00725776  0.00366694 -0.05420479 -0.0227249 ]

In [ ]:
# Count token - Use tiktoken to estimate input size and control cost.
def num_tokens_from_string(s: str, encoding_name: str = 'cl100k_base') -> int:
    enc = tiktoken.get_encoding(encoding_name)
    return len(enc.encode(s))

print('Sample token count:', num_tokens_from_string(sample))

In [ ]:
# Sample token count: 10

In [ ]:
# Dimensionality Reduction & Visualization
sentences = [
    'I love machine learning',
    'OpenAI creates powerful AI models',
    'The sky is clear today',
    'I enjoy hiking in the mountains',
    'This restaurant has great food'
]
vectors = np.vstack([get_embedding(s) for s in sentences])            # Convert each sentence into an embedding
 
pca = PCA(n_components=2)                                             ## PCA = Principal Component Analysis. 2 Dimensions . Take the many numbers in an embedding 
                                                                      # and compress them into just 2 numbers, while trying to preserve as much of the important 
                                                                      # variation/information as possible.

points = pca.fit_transform(vectors)                                   # PCA looks at all your vectors and determines the 2 most useful directions in the data.
                                                                      # It then converts every sentence into coordinates along those two directions.
"""
                  1536 dimensions
                         ↓
              ┌──────────────────┐
              │    Embeddings    │
              │                  │
              │  sentence 1      │
              │  sentence 2      │
              │  sentence 3      │
              │  sentence 4      │
              │  sentence 5      │
              └──────────────────┘
                         ↓
                        PCA
                         ↓
                 ┌──────────────┐
                 │  X     Y     │
                 │  2.4   0.8   │
                 │  2.1   1.0   │
                 │ -1.8   0.3   │
                 │ -1.2  -1.5   │
                 │ -0.9  -1.0   │
                 └──────────────┘
                         ↓
                    2D graph
"""
plt.figure(figsize=(8,6))
plt.scatter(points[:,0], points[:,1])
for i, txt in enumerate(sentences):
    plt.annotate(txt, (points[i,0], points[i,1]))
plt.title('2D PCA of Sentence Embeddings')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.show()
